Uus tabel, kus on:

pat_id (patterns.id)

head_id (transaction.head_id)

transaction_id (transaction.id)

phrase_nr (pattern.phrase_nr)

root (transaction.lemma)

Uue tabeli loomiseks:

1. transaction_head.verb matchib patterns.verb
2. transaction_head.id matchib transactions.head_id  
3. patterns.deprel + kääne peab matchima transaction.deptrel+kääne 


## Filtreerimine v2

### Juhuks kui andmeid on suurem hulk (nt isikumäärus=mitte unagi puhul) ja on vaja töödelda andmeid jupikaupa

In [ ]:
import sqlite3

In [2]:
# verbimustrite andmebaas
#pattern_db = "../example_data/verb_patterns_actors.db"

# transaktsioonide andmebaas
transaction_db = "../example_data/transactions.db"

# Siia salvestuvad loodavad tabelid
vp_data_db = "../example_data/vp_data_actors.db"

transactions_table = "transaction_v2" # "trans.'transaction'"

# state of the verb (isikumäärus)
# alati: alati isikumäärus etc
STAT = 'alati' #'mitte_kunagi'

In [3]:
con = sqlite3.connect(vp_data_db)
cur = con.cursor()

In [4]:
# transaktsioonide andmebaasi lisamine
cur.execute(f'ATTACH DATABASE "{transaction_db}" AS trans')

#### since there are too many patterns for local computer to handle, then the patterns_* table is divided into n parts and the following steps are repeated n times

In [ ]:
n = 15

In [ ]:
cur.execute("""
DROP TABLE IF EXISTS patterns_actors_{stat}_help
""".format(stat=STAT))

# ntile -> n parts
cur.execute("""
create table patterns_actors_{stat}_help as
select *,NTILE(n) OVER(ORDER BY pat_id) as batch_nbr
from patterns_actors_{stat2};
""".format(stat=STAT, n=n, stat2=STAT))

In [ ]:
tables = cur.execute("""
SELECT batch_nbr, count(*) as cnt FROM patterns_actors_{stat}_help
group by batch_nbr
""".format(stat=STAT))

for i, elem in enumerate(tables):
    print(elem)

### Luua uus tabel pat_tr_head_*, kus on:

head_id, pat_id, verb_word, phrase_nr, phrase_case, deprel (verbi deprel)

Tabeli loomine:

Teha join transaction_head tabeliga verbi alusel.


In [ ]:
%%time

# repeat n times and change the number at the end of the table name and batch_nbr
num = 1

cur.execute("""
DROP TABLE IF EXISTS pat_tr_head_{stat}_{num}
""".format(stat=STAT, num=num))

cur.execute("""
CREATE TABLE pat_tr_head_{stat}_{num1} AS
SELECT DISTINCT
    head.id as head_id,
    pat.pat_id as pat_id,
    pat.verb_word as verb_word,
    pat.phrase_nr as phrase_nr,
    pat.phrase_case as phrase_case,
    pat.pat_deprel as pat_deprel
FROM 
    patterns_actors_{stat2}_help as pat
INNER JOIN 
    transaction_head as head
ON
    pat.verb_word = head.verb
WHERE 
    pat.batch_nbr = {num}

""".format(stat=STAT, num1=num, stat2=STAT, num=num))

### Luua uus tabel patterns_transaction_actors_*, kus on:

head_id, pat_id, transaction_id, phrase_nr, verb_word, root_word, verb_deprel, word_deprel, pos, phrase_case

Join toimub pat_tr_head_v1 ja transaction_v2 vahel. 

Praegu on joini aluseks head_id, deprel ja feats. 

` NB!!! patterns tabeli kääne on abl/all/ad jne ja neid tulebks matchida feats veerus olevaga`

In [ ]:
%%time 

# repeat n times and change the number at the end of the table names
num = 1

cur.execute("""
DROP TABLE IF EXISTS patterns_transaction_actors_{stat}_2_{num}
""".format(stat=STAT, num=num))

cur.execute("""
CREATE TABLE patterns_transaction_actors_{stat}_2_{num1} AS
SELECT DISTINCT
    tbl1.head_id as head_id,
    tbl1.pat_id as pat_id,
    tr.id as transaction_id,
    tbl1.phrase_nr as phrase_nr,
    tbl1.verb_word as verb_word,
    tr.lemma as root_word,
    tbl1.pat_deprel as pat_deprel,
    tr.deprel as word_deprel,
    tr.pos as pos,
    tbl1.phrase_case as phrase_case,
    tr.feats as tr_feats,
    tr.koht as koht,
    tr.elus as elus
    
FROM 
    pat_tr_head_{stat2}_{num2} as tbl1
JOIN 
    {trans_tbl} as tr
ON 
    tbl1.head_id = tr.head_id
    and tbl1.pat_deprel = tr.deprel
WHERE INSTR(',' || tr.feats || ',', ',' || tbl1.phrase_case || ',') > 0

""".format(stat=STAT, num1=num, stat2=STAT, num2=num, trans_tbl=transactions_table))

### compile tables

In [ ]:
%%time 

cur.execute("""
DROP TABLE IF EXISTS patterns_transaction_actors_{stat}_comp_temp
""".format(stat=STAT))

cur.execute("""
CREATE TABLE patterns_transaction_actors_{stat}_comp_temp AS
SELECT * FROM patterns_transaction_actors_{stat2}_2_1
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_2
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_3
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_4
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_5
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_6
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_7
""".format(stat=STAT, stat2=STAT))
con.commit()

cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_8
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_9
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_10
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_11
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_12
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_13
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_14
""".format(stat=STAT, stat2=STAT))
con.commit()
cur.execute("""
INSERT INTO patterns_transaction_actors_{stat}_comp_temp
SELECT * FROM patterns_transaction_actors_{stat2}_2_15
""".format(stat=STAT, stat2=STAT))

con.commit()

cur.execute("""
DROP TABLE IF EXISTS patterns_transaction_actors_{stat}_2
""".format(stat=STAT))

cur.execute("""
CREATE TABLE patterns_transaction_actors_{stat}_2 AS
SELECT DISTINCT * FROM patterns_transaction_actors_{stat2}_comp_temp
""".format(stat=STAT, stat2=STAT))

con.commit()

In [ ]:
con.close()